In [2]:
from dask.distributed import Client, SSHCluster
import dask.bag as db
import dask
import glob
import numpy as np
import matplotlib.pyplot as plt
from kafka import KafkaConsumer
from kafka.admin import KafkaAdminClient, NewTopic
from kafka.errors import UnknownTopicOrPartitionError

In [16]:
BOOTSTRAP = "10.67.22.102:19092" # Kafka port 19092 is dedicated to VMs in CloudVeneto
TOPIC_STREAM = "topic_stream"
TOPIC_RESULTS = "topic_results"
N_WORKERS = 2

kafka_admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)

def create_topic(topic_name, partitions):

    kafka_admin.delete_topics([topic_name])

    kafka_admin.create_topics([
        NewTopic(
            name=topic_name,
            num_partitions=partitions,
            replication_factor=1
        )
    ])
    
    print(f"Created topic '{topic_name}' with {partitions} partitions.")

create_topic(TOPIC_STREAM, N_WORKERS)
create_topic(TOPIC_RESULTS, 1)
print("\nList of topics:",kafka_admin.list_topics())

Created topic 'topic_stream' with 2 partitions.
Created topic 'topic_results' with 1 partitions.

List of topics: ['topic_stream', '__consumer_offsets']


In [17]:
cluster = SSHCluster(
    [
        "10.67.22.102",   # scheduler IP
        "10.67.22.58",   # worker1 IP
        "10.67.22.244"   # worker2 IP
    ],
    connect_options={"known_hosts": None},
    scheduler_options={"port": 8786, "dashboard_address": "0.0.0.0:8797"},
    worker_options={"n_workers": 1, "nthreads": 2}
)

client = Client(cluster)
client

2026-07-11 16:36:26,672 - distributed.deploy.ssh - INFO - 2026-07-11 16:36:26,669 - distributed.http.proxy - INFO - To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
2026-07-11 16:36:26,691 - distributed.deploy.ssh - INFO - 2026-07-11 16:36:26,691 - distributed.scheduler - INFO - State start
2026-07-11 16:36:26,694 - distributed.deploy.ssh - INFO - 2026-07-11 16:36:26,694 - distributed.scheduler - INFO -   Scheduler at:   tcp://10.67.22.102:8786
2026-07-11 16:36:27,195 - distributed.deploy.ssh - INFO - 2026-07-11 16:36:27,192 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.244:43185'
2026-07-11 16:36:27,400 - distributed.deploy.ssh - INFO - 2026-07-11 16:36:27,396 - distributed.nanny - INFO -         Start Nanny at: 'tcp://10.67.22.58:45805'
2026-07-11 16:36:27,503 - distributed.deploy.ssh - INFO - 2026-07-11 16:36:27,499 - distributed.diskutils - INFO - Found stale lock file and directory '/t

<Client: 'tcp://10.67.22.102:8786' processes=1 threads=2, memory=3.83 GiB>

In [18]:
# to see the dask dashboard: http://localhost:8797/status

In [19]:
def process_batch(bytestring):
    SAMPLES_PER_SCAN = 2**11
    FREQ = 2.0 * 1e6
    
    half_length = len(bytestring) // 2
    i_data = np.frombuffer(bytestring[:half_length], dtype="<f4")
    q_data = np.frombuffer(bytestring[half_length:], dtype="<f4")

    complex_signal = np.empty(i_data.shape, dtype=np.complex64)
    complex_signal.real = i_data
    complex_signal.imag = q_data

    n_scans = len(complex_signal) // SAMPLES_PER_SCAN
    matrix = complex_signal.reshape((n_scans, SAMPLES_PER_SCAN))

    ffts = np.fft.fftshift(np.fft.fft(matrix, axis=1), axes=1)
    power = np.abs(ffts)**2

    avg_power = np.mean(power, axis=0)
    M2 = ((power - avg_power)**2).sum(axis=0)
    freqs = np.fft.fftshift(np.fft.fftfreq(SAMPLES_PER_SCAN, d=1/FREQ))
    
    return {
        "n_scans": n_scans,
        "frequency": freqs,
        "mean": avg_power,
        "M2": M2
    }

def worker_kafka_processor(worker_id, bootstrap_servers, topic_in, topic_out):
    PUBLISH_INTERVAL_SEC = 5.0

    from kafka import KafkaConsumer, KafkaProducer
    import json
    import numpy as np
    import time
    
    consumer = KafkaConsumer(
        topic_in,
        bootstrap_servers=bootstrap_servers,
        group_id='dask_processing_group', 
        auto_offset_reset='latest'
    )
    
    producer = KafkaProducer(
        bootstrap_servers=bootstrap_servers,
        value_serializer=lambda v: json.dumps(v).encode('utf-8')
    )

    accumulated_mean = None
    accumulated_M2 = None
    accumulated_n = 0
    frequency = None
    batch_latencies_ms = []
    producer_tss = []

    publish_deadline = time.monotonic() + PUBLISH_INTERVAL_SEC
    
    for msg in consumer:
        for key, value in msg.headers:
            if key == "producer_ts":
                producer_ts = float(value.decode('utf-8'))
                break
        producer_tss.append(producer_ts)

        t_start = time.perf_counter()
        res = process_batch(msg.value)

        batch_n = res["n_scans"]
        batch_mean = res["mean"]
        batch_M2 = res["M2"]

        if accumulated_mean is None:
            accumulated_mean = batch_mean.copy()
            accumulated_M2 = batch_M2.copy()
            frequency = res["frequency"]
            accumulated_n = batch_n
        else:
            delta = batch_mean - accumulated_mean
            total_n = accumulated_n + batch_n
            accumulated_mean = accumulated_mean + delta * batch_n / total_n
            accumulated_M2 = accumulated_M2 + batch_M2 + delta**2 * accumulated_n * batch_n / total_n
            accumulated_n = total_n

        t_end = time.perf_counter()
        batch_latencies_ms.append((t_end - t_start) * 1000)

        now = time.monotonic()
        if now >= publish_deadline and accumulated_n > 0:
            result = {
                "worker_id": worker_id,
                "num_averaged_scans": int(accumulated_n),
                "average": {
                    "frequency": frequency.tolist(),
                    "power_mean": accumulated_mean.tolist(),
                    "power_M2": accumulated_M2.tolist()
                },
                "metrics": {
                    "producer_tss": producer_tss,
                    "fft_latencies_ms": batch_latencies_ms
                }
            }

            producer.send(topic_out, value=result)
            producer.flush()

            accumulated_mean = None
            accumulated_M2 = None
            accumulated_n = 0
            frequency = None
            batch_latencies_ms = []
            producer_tss = []

            publish_deadline = time.monotonic() + PUBLISH_INTERVAL_SEC

In [20]:
res1 = client.submit(
    worker_kafka_processor, 
    worker_id="Worker1", 
    bootstrap_servers=BOOTSTRAP, 
    topic_in=TOPIC_STREAM, 
    topic_out=TOPIC_RESULTS, 
    workers="10.67.22.58"
)

res2 = client.submit(
    worker_kafka_processor, 
    worker_id="Worker2", 
    bootstrap_servers=BOOTSTRAP, 
    topic_in=TOPIC_STREAM, 
    topic_out=TOPIC_RESULTS, 
    workers="10.67.22.244"
)

print(res1.status)
print(res2.status)

pending
pending


In [15]:
client.close()
cluster.close()